## Pre-processing & Feature Engineering: E-Commerce Customer Behavior
**Role:** Data Engineer (Ibnu Dwiki Hermawan)

**Tujuan Notebook:**

1. Membersihkan data mentah dari *missing values* dan anomali transaksi.

2. Mengekstrak fitur RFM dan fitur tambahan (AvgSpending, UniqueProducts).

3. Menggunakan DBSCAN untuk menyaring *noise* / *outliers* sebelum masuk ke pemodelan K-Means.

In [ ]:
# Import library esensial
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from datetime import timedelta

# Konfigurasi visualisasi
sns.set_theme(style="whitegrid")
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Memuat dataset (Pastikan path file disesuaikan dengan folder struktur repositori)
# Jika diletakkan di folder 'data/raw/':
file_path = '../data/raw/data.csv'
df = pd.read_csv(file_path, encoding="ISO-8859-1")

print(f"Dimensi data awal: {df.shape}")
df.head()

## 1. Data Cleaning
Menghapus baris tanpa `CustomerID`, transaksi dengan kuantitas negatif/nol, dan transaksi yang dibatalkan (berawalan huruf 'C').

In [ ]:
# Hapus missing values pada CustomerID
df_clean = df.dropna(subset=['CustomerID']).copy()

# Hapus transaksi batal dan kuantitas negatif
df_clean['InvoiceNo'] = df_clean['InvoiceNo'].astype(str)
df_clean = df_clean[~df_clean['InvoiceNo'].str.startswith('C')]
df_clean = df_clean[df_clean['Quantity'] > 0]

print(f"Dimensi data setelah dibersihkan: {df_clean.shape}")

## 2. Feature Engineering (RFM + Fitur Tambahan)
Menghitung Recency, Frequency, dan Monetary, serta menambahkan `UniqueProducts` (keragaman produk) dan `AvgSpending` (rata-rata pengeluaran per transaksi).

In [ ]:
# Konversi tanggal dan hitung TotalPrice
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']

# Tanggal referensi untuk Recency (1 hari setelah transaksi terakhir)
snapshot_date = df_clean['InvoiceDate'].max() + timedelta(days=1)

# Agregasi tingkat Customer
customer_df = df_clean.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days, # Recency
    'InvoiceNo': 'nunique',                                  # Frequency
    'TotalPrice': ['sum', 'mean'],                           # Monetary & AvgSpending
    'StockCode': 'nunique'                                   # UniqueProducts
})

# Merapikan nama kolom multi-level
customer_df.columns = ['Recency', 'Frequency', 'Monetary', 'AvgSpending', 'UniqueProducts']
customer_df.head()

## 3. Scaling Data
Algoritma DBSCAN sangat sensitif terhadap skala jarak. Kita perlu menstandarisasi fitur menggunakan `StandardScaler`.

In [ ]:
scaler = StandardScaler()
scaled_features = scaler.fit_transform(customer_df)
scaled_df = pd.DataFrame(scaled_features, columns=customer_df.columns, index=customer_df.index)

scaled_df.head()

## 4. DBSCAN untuk Filter Noise
Mendeteksi pelanggan yang memiliki pola transaksi anomali (*outliers*) agar tidak merusak klaster utama pada algoritma K-Means nanti.

In [ ]:
# Inisialisasi dan fit model DBSCAN
# (Parameter eps dan min_samples bisa di-tuning lebih lanjut)
dbscan = DBSCAN(eps=1.5, min_samples=5)
customer_df['Cluster_DBSCAN'] = dbscan.fit_predict(scaled_df)

# Memisahkan Inliers (bersih) dan Outliers (noise)
clean_data = customer_df[customer_df['Cluster_DBSCAN'] != -1].copy()
noise_data = customer_df[customer_df['Cluster_DBSCAN'] == -1].copy()

# Buang kolom penanda DBSCAN dari data bersih
clean_data = clean_data.drop(columns=['Cluster_DBSCAN'])

print(f"Jumlah Pelanggan Normal (Inliers): {len(clean_data)}")
print(f"Jumlah Pelanggan Anomali (Outliers): {len(noise_data)}")

## 5. Export Data
Menyimpan dataset yang sudah bersih untuk diteruskan ke tahap K-Means.

In [ ]:
# Ekspor ke folder processed
clean_data.to_csv('../data/processed/clean_customer_features.csv')
noise_data.to_csv('../data/processed/anomalous_customers.csv')

print("Proses Data Engineering selesai. File CSV berhasil disimpan!")